In [2]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import numpy as np
import pandas as pd
from common.const import DEVICE, SCHEMA, TABLE_RAW_PRODUCTS, TABLE_RAW_REVIEWS
from common.helper import clickhouse_extract_data_as_df, connect_qdrant, connect_clickhouse
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

In [3]:
# Query data

def query_products_data(n_rows, offset):
    products_query = f"""
    SELECT
        parent_asin,
        title,
        description,
        features,
        main_category,
        details,
        CAST(price AS Nullable(Float64)) AS price,
        average_rating,
        rating_number
    FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
    LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(products_query)

def query_reviews_data(n_rows, offset):
    reviews_query = f"""
    SELECT 
        user_id,
        parent_asin,
        title,
        text,
        rating
    FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
    LIMIT {n_rows} OFFSET {offset}
    """
    return clickhouse_extract_data_as_df(reviews_query)

In [4]:
def create_product_text(row):
    text_parts = []

    if pd.notna(row['title']):
        text_parts.append((row['title'] + " ") * 1 + row['title']) # duplicate once

    if pd.notna(row['description']):
        text_parts.append(row['description'])

    if pd.notna(row['features']):
        text_parts.append(row['features'])

    if pd.notna(row['main_category']):
        text_parts.append((row['main_category'] + " ") * 2 + row['main_category'])
    
    if pd.notna(row['details']):
        text_parts.append(row['details'])

    return ' '.join(text_parts)

In [ ]:
# reviews_df['full_review'] = (
#     reviews_df['title'].fillna('') + ' ' +
#     reviews_df['text'].fillna('')
# )

# user_reviews = (
#     reviews_df
#     .groupby('user_id')['full_review']
#     .apply(' '.join)
#     .reset_index(name='all_reviews_text')
# )

In [6]:
import time
import numpy as np
from sentence_transformers import SentenceTransformer
import re
# ---------------------------------------------------------------------------
# Helper: DDL / INSERT via your existing ClickHouse client
# Adapt `clickhouse_execute` to however your project runs non-SELECT queries.
# ---------------------------------------------------------------------------
def clickhouse_execute(query: str, data=None):
    """
    Fixed for clickhouse_connect: DDL → command(), INSERT → insert_df()
    """
    client = connect_clickhouse()
    try:
        if data is not None:
            # INSERT: convert to DataFrame and use insert_df (handles Array(Float32))
            df = pd.DataFrame(data, columns=['row_id', 'embedding'])
            # Auto-extract table name from query
            tbl_match = re.search(r'INSERT INTO (\S+)', query)
            if not tbl_match:
                raise ValueError(f"Could not parse table name from: {query}")
            tbl_name = tbl_match.group(1)
            client.insert_df(tbl_name, df)
        else:
            # DDL/CREATE/ALTER/DROP
            client.command(query)
    finally:
        client.close()


# ---------------------------------------------------------------------------
# ClickHouse table helpers
# ---------------------------------------------------------------------------
TABLE_EXACT = "{schema}.products_vectors_exact_{size}"
TABLE_HNSW  = "{schema}.products_vectors_hnsw_{size}"

def create_exact_table(schema, dataset_size, vector_dim):
    """Brute-force cosineDistance table (no ANN index)."""
    tbl = TABLE_EXACT.format(schema=schema, size=dataset_size)
    clickhouse_execute(f"""
        CREATE TABLE IF NOT EXISTS {tbl}
        (
            row_id    Int64,
            embedding Array(Float32) CODEC(NONE),
            CONSTRAINT chk_dim CHECK length(embedding) = {vector_dim}
        )
        ENGINE = MergeTree
        ORDER BY row_id
    """)
    return tbl

def create_hnsw_table(schema, dataset_size, vector_dim, m, ef_construction, ef_search):
    """Approximate HNSW vector similarity table."""
    tbl = TABLE_HNSW.format(schema=schema, size=dataset_size)
    clickhouse_execute(f"""
        CREATE TABLE IF NOT EXISTS {tbl}
        (
            row_id    Int64,
            embedding Array(Float32) CODEC(NONE),
            CONSTRAINT chk_dim CHECK length(embedding) = {vector_dim},
            INDEX idx_embedding embedding
                TYPE vector_similarity(
                    'hnsw', 'cosineDistance', {vector_dim},
                    'bf16',          -- quantization (bf16 default, matches IndexHNSWSQ 8-bit spirit)
                    {m},             -- hnsw_max_connections_per_layer  (== FAISS m)
                    {ef_construction}-- hnsw_candidate_list_size_for_construction (== FAISS ef_construction)
                )
        )
        ENGINE = MergeTree
        ORDER BY row_id
        SETTINGS index_granularity = 8192
    """)
    return tbl

def drop_table(tbl):
    clickhouse_execute(f"DROP TABLE IF EXISTS {tbl}")

def insert_embeddings(tbl, row_ids, embeddings, batch_size=500):
    """Batch-insert (row_id, embedding) pairs into a ClickHouse table."""
    for i in range(0, len(embeddings), batch_size):
        batch_ids   = row_ids[i : i + batch_size]
        batch_vecs  = embeddings[i : i + batch_size]
        data = [
            (int(rid), vec.tolist())
            for rid, vec in zip(batch_ids, batch_vecs)
        ]
        clickhouse_execute(
            f"INSERT INTO {tbl} (row_id, embedding) VALUES",
            data=data
        )

def search_exact(tbl, query_vector: np.ndarray, k: int) -> list[int]:
    """Linear-scan cosineDistance search — equivalent to FAISS IndexFlatIP."""
    vec_list = query_vector.tolist()
    rows = clickhouse_extract_data_as_df(f"""
        WITH {vec_list} AS ref_vec
        SELECT row_id
        FROM {tbl}
        ORDER BY cosineDistance(embedding, ref_vec) ASC
        LIMIT {k}
    """)
    return rows["row_id"].tolist()

def search_hnsw(tbl, query_vector: np.ndarray, k: int,
                ef_search: int = 256) -> list[int]:
    """
    Approximate HNSW search — equivalent to FAISS IndexHNSWSQ.search().
    ef_search maps directly to hnsw_candidate_list_size_for_search.
    """
    vec_list = query_vector.tolist()
    rows = clickhouse_extract_data_as_df(f"""
        WITH {vec_list} AS ref_vec
        SELECT row_id
        FROM {tbl}
        ORDER BY cosineDistance(embedding, ref_vec) ASC
        LIMIT {k}
        SETTINGS hnsw_candidate_list_size_for_search = {ef_search}
    """)
    return rows["row_id"].tolist()


# ---------------------------------------------------------------------------
# Precision@K and stats helpers (unchanged)
# ---------------------------------------------------------------------------
def precision_at_k(I_exact, I_approx):
    matches = [
        len(set(e).intersection(set(a)))
        for e, a in zip(I_exact, I_approx)
    ]
    return np.mean(np.array(matches) / len(I_exact[0]))

def mean_pm_2std(values):
    values = np.array(values)
    mean   = np.mean(values)
    std    = np.std(values, ddof=1)
    return mean, std, mean - 2 * std, mean + 2 * std


# ---------------------------------------------------------------------------
# Main evaluation loop  (drop-in replacement for the FAISS block)
# ---------------------------------------------------------------------------
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

product_asins = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 
                 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']

BATCH_SIZE      = 5000
k               = 11          # fetch k; first result is self-match → remove it

for dataset_size in [500_000]:
    # ── 1. Embed corpus ────────────────────────────────────────────────
    products_df = query_products_data(dataset_size, 0)
    products_df['combined_text'] = products_df.apply(create_product_text, axis=1)
    product_embeddings = model.encode(
        products_df['combined_text'].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True,
        device=DEVICE
    ).astype("float32")
    np.save("product_embeddings.npy", product_embeddings)
    # product_embeddings = np.load("product_embeddings.npy")


    # ── 1. Prepare & embed user corpus ─────────────────────────────────────────
    reviews_df = query_reviews_data(dataset_size, 0)
    reviews_df['full_review'] = (
        reviews_df['title'].fillna('') + ' ' + reviews_df['text'].fillna('')
    )
    user_reviews = (
        reviews_df.groupby('user_id')['full_review']
        .apply(' '.join)
        .reset_index(name='all_reviews_text')
    )
    user_embeddings = model.encode(
        user_reviews['all_reviews_text'].tolist(),
        show_progress_bar=True,
        batch_size=32,
        normalize_embeddings=True,
        device=DEVICE
    ).astype("float32")
    np.save("user_embeddings.npy", user_embeddings)
    user_embeddings = np.load("user_embeddings.npy")

    for vector_dim in [384]:
        for m in [8, 16, 32]:
            if m == 8:
                ef_construction = 50
                ef_search = 32
            elif m == 16:
                ef_construction = 100
                ef_search = 64
            elif m == 32:
                ef_construction = 200
                ef_search = 128
            print(f"\n=== dataset_size={dataset_size}, vector_dim={vector_dim} ===")

            # ── 2. Create ClickHouse tables (fresh) ───────────────────────────
            tbl_exact = create_exact_table(SCHEMA, dataset_size, vector_dim)
            tbl_hnsw  = create_hnsw_table(SCHEMA, dataset_size, vector_dim,
                                        m, ef_construction, ef_search)

            # ── 3. Upload embeddings ───────────────────────────────────────────
            row_ids = products_df.index.to_numpy().astype("int64")

            print("Uploading to exact table …")
            insert_embeddings(tbl_exact, row_ids, product_embeddings, BATCH_SIZE)

            print("Uploading to HNSW table …")
            insert_embeddings(tbl_hnsw, row_ids, product_embeddings, BATCH_SIZE)
            print("finish upload hnws")
            # Optionally force index materialisation on the HNSW table
            # (ClickHouse builds the index automatically on insert,
            #  but MATERIALIZE ensures it if deferred)
            clickhouse_execute(
                f"ALTER TABLE {tbl_hnsw} MATERIALIZE INDEX idx_embedding "
                f"SETTINGS mutations_sync = 2"
            )

            # ── 4. Embed test queries ──────────────────────────────────────────
            test_query = f"""
                SELECT parent_asin, title, description, features,
                    main_category, details,
                    CAST(price AS Nullable(Float64)) AS price,
                    average_rating, rating_number
                FROM {SCHEMA}.{TABLE_RAW_PRODUCTS}
                WHERE parent_asin IN {tuple(product_asins)}
            """
            test_products_df = clickhouse_extract_data_as_df(test_query)
            test_products_df['combined_text'] = test_products_df.apply(
                create_product_text, axis=1)
            test_product_embeddings = model.encode(
                test_products_df['combined_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,
                device=DEVICE
            ).astype("float32")

            # ── 5. Run timed searches and compute Precision@10 ─────────────────
            exact_times = []
            approx_times = []
            p10s = []

            for test_emb in test_product_embeddings:
                query_vec = test_emb.reshape(1, -1)

                # Exact (linear scan)
                t0 = time.time()
                ids_exact = search_exact(tbl_exact, test_emb, k)
                exact_times.append(time.time() - t0)

                # Approximate (HNSW)
                t0 = time.time()
                ids_hnsw = search_hnsw(tbl_hnsw, test_emb, k, ef_search)
                approx_times.append(time.time() - t0)

                # Remove self-match (first result) — same as I[:, 1:] in FAISS
                ids_exact_no_self = ids_exact[1:]
                ids_hnsw_no_self  = ids_hnsw[1:]

                p10s.append(precision_at_k([ids_exact_no_self],
                                            [ids_hnsw_no_self]))

            # ── 6. Report ──────────────────────────────────────────────────────
            for label, times in [("Exact", exact_times), ("Approx", approx_times)]:
                mn, sd, lo, hi = mean_pm_2std(times)
                print(f"PRODUCT {dataset_size} {label} Time: "
                    f"{mn:.4f} ± {2*sd:.4f}  (range: {lo:.4f} - {hi:.4f})")

            p_mn, p_sd, p_lo, p_hi = mean_pm_2std(p10s)
            print(f"PRODUCT {dataset_size} Precision@10: "
                f"{p_mn:.4f} ± {2*p_sd:.4f}  (range: {p_lo:.4f} - {p_hi:.4f})")
            
                        

            # ── 2. Create ClickHouse tables ────────────────────────────────────────────
            tbl_user_exact = f"{SCHEMA}.user_vectors_exact_{dataset_size}"
            tbl_user_hnsw  = f"{SCHEMA}.user_vectors_hnsw_{dataset_size}_{m}_{ef_construction}_{ef_search}"

            drop_table(tbl_user_exact)
            drop_table(tbl_user_hnsw)

            tbl_user_exact = create_exact_table(SCHEMA, f"user_{dataset_size}", vector_dim)
            tbl_user_hnsw  = create_hnsw_table(SCHEMA, f"user_{dataset_size}", vector_dim,
                                                m, ef_construction, ef_search)

            # ── 3. Upload user embeddings ──────────────────────────────────────────────
            user_row_ids = user_reviews.index.to_numpy().astype("int64")
            insert_embeddings(tbl_user_exact, user_row_ids, user_embeddings, BATCH_SIZE)
            insert_embeddings(tbl_user_hnsw, user_row_ids, user_embeddings, BATCH_SIZE)

            # Force HNSW index materialization
            clickhouse_execute(f"""
                ALTER TABLE {tbl_user_hnsw} MATERIALIZE INDEX idx_embedding
                SETTINGS mutations_sync = 2
            """)

            # ── 4. Prepare test queries ────────────────────────────────────────────────
            random_users = ['AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA', 'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 
                            'AHT6VS3G3JSA7KBOCBT6KPUCJX7A', 'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA', 
                            'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA', 'AHA272AOVKWZUXL672G653A2MTIA', 
                            'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ']

            test_user_query = f"""
                SELECT user_id, parent_asin, title, text, rating
                FROM {SCHEMA}.{TABLE_RAW_REVIEWS}
                WHERE user_id IN {tuple(random_users)}
            """
            test_reviews_df = clickhouse_extract_data_as_df(test_user_query)
            test_reviews_df['full_review'] = (
                test_reviews_df['title'].fillna('') + ' ' +
                test_reviews_df['text'].fillna('')
            )
            test_user_reviews = (
                test_reviews_df.groupby('user_id')['full_review']
                .apply(' '.join)
                .reset_index(name='all_reviews_text')
            )
            test_user_embeddings = model.encode(
                test_user_reviews['all_reviews_text'].tolist(),
                show_progress_bar=True,
                batch_size=32,
                normalize_embeddings=True,
                device=DEVICE
            ).astype("float32")

            # ── 5. Benchmark searches ──────────────────────────────────────────────────
            k = 11
            exact_times, approx_times, p10s = [], [], []

            for test_emb in test_user_embeddings:
                # Exact linear scan
                t0 = time.time()
                ids_exact = search_exact(tbl_user_exact, test_emb, k)
                exact_times.append(time.time() - t0)

                # HNSW approximate
                t0 = time.time()
                ids_hnsw = search_hnsw(tbl_user_hnsw, test_emb, k, ef_search)
                approx_times.append(time.time() - t0)

                # Precision@10 (remove self-match)
                ids_exact_no_self = ids_exact[1:]
                ids_hnsw_no_self  = ids_hnsw[1:]
                p10s.append(precision_at_k([ids_exact_no_self], [ids_hnsw_no_self]))

            # ── 6. Report results ──────────────────────────────────────────────────────
            exact_mean, exact_std, exact_low, exact_high = mean_pm_2std(exact_times)
            approx_mean, approx_std, approx_low, approx_high = mean_pm_2std(approx_times)
            p10_mean, p10_std, p10_low, p10_high = mean_pm_2std(p10s)

            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} "
                f"Precision@10: {p10_mean:.4f} ± {2*p10_std:.4f} "
                f"(range: {p10_low:.4f} - {p10_high:.4f})")

            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} "
                f"Exact Time: {exact_mean:.4f} ± {2*exact_std:.4f} "
                f"(range: {exact_low:.4f} - {exact_high:.4f})")

            print(f"USER {dataset_size} {m} {ef_construction} {ef_search} "
                f"Approx Time: {approx_mean:.4f} ± {2*approx_std:.4f} "
                f"(range: {approx_low:.4f} - {approx_high:.4f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


connected to clickhouse


Batches:   0%|          | 0/15625 [00:00<?, ?it/s]

connected to clickhouse


...

connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
USER 500000 16 100 64 Precision@10: 0.7200 ± 0.5797 (range: 0.1403 - 1.2997)
USER 500000 16 100 64 Exact Time: 0.4773 ± 0.0647 (range: 0.4127 - 0.5420)
USER 500000 16 100 64 Approx Time: 0.2393 ± 1.0306 (range: -0.7913 - 1.2699)

=== dataset_size=500000, vector_dim=384 ===
Uploading to exact table …
Uploading to HNSW table …
finish upload hnws
connected to clickhouse


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
PRODUCT 500000 Exact Time: 0.6375 ± 0.1032  (range: 0.5343 - 0.7408)
PRODUCT 500000 Approx Time: 0.2774 ± 1.1761  (range: -0.8986 - 1.4535)
PRODUCT 500000 Precision@10: 0.7700 ± 0.5892  (range: 0.1808 - 1.3592)
connected to clickhouse


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
connected to clickhouse
USER 500000 32 200 128 Precision@10: 0.7600 ± 0.5432 (range: 0.2168 - 1.3032)
USER 500000 32 200 128 Exact Time: 0.7969 ± 1.5208 (range: -0.7239 - 2.3177)
USER 500000 32 200 128 Approx Time: 0.2652 ± 1.1003 (range: -0.8351 - 1.3655)
